# 2. Cleaning

## Set up

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

Project root: /Users/Local/REPOSITORIES/injury_risk_classifier
Current working directory: /Users/Local/REPOSITORIES/injury_risk_classifier


In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
import os 

%load_ext autoreload
%autoreload 2

from cleaning_and_features.clean import *

In [3]:
data_path = PROJECT_ROOT / "data" / "interim" / "raw_modeling_data.csv"

df = pd.read_csv(data_path)

## Cleaning

Code for this is in `clean.py`, but I walk through with the dataset 

In [4]:
__, miss_report = explore(df, missing = True, threshold= 75)

inspecting data...

There are 282358 rows and 22 columns
There 71 duplicate rows



In [5]:
df = df.drop_duplicates()

In [6]:
print(miss_report)

                              data_types  missing_count  missing_percent  \
top_traffic_accident_offense         str              0             0.00   
first_occurrence_date            float64              0             0.00   
incident_address                     str              0             0.00   
geo_lon                          float64          11155             3.95   
geo_lat                          float64          11155             3.95   
TU1_DRIVER_ACTION                    str          13423             4.75   
TU2_DRIVER_ACTION                    str          15146             5.36   
TU1_VEHICLE_TYPE                     str           6849             2.43   
TU2_VEHICLE_TYPE                     str          10038             3.56   
TU1_DRIVER_HUMANCONTRIBFACTOR        str          15813             5.60   
TU2_DRIVER_HUMANCONTRIBFACTOR        str          14182             5.02   
TU1_PEDESTRIAN_ACTION                str          77431            27.42   
SERIOUSLY_IN

### Date columns

Turns the julian `first_occurance_date` into two columns, time and date

In [7]:
df = split_datetime(df, "first_occurrence_date")

In [8]:
df.head()

,top_traffic_accident_offense,first_occurrence_date,incident_address,geo_lon,geo_lat,TU1_DRIVER_ACTION,TU2_DRIVER_ACTION,TU1_VEHICLE_TYPE,TU2_VEHICLE_TYPE,TU1_DRIVER_HUMANCONTRIBFACTOR,...,FATALITY_MODE_2,SERIOUSLY_INJURED_MODE_1,SERIOUSLY_INJURED_MODE_2,TU2_PEDESTRIAN_ACTION,ROAD_DESCRIPTION,ROAD_CONDITION,LIGHT_CONDITION,datetime,date,time
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,No Contributing Action,No Contributing Action,Pickup Truck/Utility Van,Transit Bus,No Apparent Contributing Factor,...,NaN,NaN,NaN,NaN,Non-Intersection,Dry,Daylight,2022-12-09 02:05:59.999991956,2022-12-09,02:05:59
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,Failed to Yield ROW,No Contributing Action,Passenger Car/Passenger Van,SUV,No Apparent Contributing Factor,...,NaN,NaN,NaN,NaN,At Intersection,Dry,Daylight,2022-12-11 23:39:59.999991056,2022-12-11,23:39:59
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,Followed Too Closely,No Contributing Action,Passenger Car/Passenger Van,Passenger Car/Passenger Van,No Apparent Contributing Factor,...,NaN,NaN,NaN,NaN,Non-Intersection,Dry,Daylight,2022-12-12 00:10:00.000004471,2022-12-12,00:10:00
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,Followed Too Closely,No Contributing Action,Pickup Truck/Utility Van,Passenger Car/Passenger Van,Aggressive Driving,...,NaN,NaN,NaN,NaN,Non-Intersection,Dry,Dark-Lighted,2022-12-09 06:26:59.999983903,2022-12-09,06:26:59
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,NaN,NaN,Unknown (Hit and Run Only),Passenger Car/Passenger Van,NaN,...,NaN,NaN,NaN,NaN,Non-Intersection,Dry,Dark-Lighted,2022-12-09 07:30:00.000000000,2022-12-09,07:30:00


### Text columns

In [9]:
col = ['TU1_VEHICLE_TYPE', 'TU2_VEHICLE_TYPE', 'TU1_DRIVER_ACTION', 'TU2_DRIVER_ACTION', 
       'TU1_DRIVER_HUMANCONTRIBFACTOR', 'TU2_DRIVER_HUMANCONTRIBFACTOR', 
       'TU1_PEDESTRIAN_ACTION', "ROAD_DESCRIPTION", "ROAD_CONDITION", "LIGHT_CONDITION"]

df = clean_text_colummns(df, col)

### Column names

In [10]:
df = clean_column_names(df)

### Create Outcome

In [11]:
df = create_outcome(df)

### Dropping columns

In [12]:
sub = df.loc[:, ["fatality_mode_1", "fatality_mode_2",
            "seriously_injured_mode_1", "seriously_injured_mode_2"]]

In [13]:
__, miss = explore(sub, missing=True)

inspecting data...

There are 282287 rows and 4 columns
There 282081 duplicate rows



In [14]:
print(miss)

                         data_types  missing_count  missing_percent  \
fatality_mode_1                 str          78499            27.81   
fatality_mode_2                 str          78660            27.87   
seriously_injured_mode_1        str          77068            27.30   
seriously_injured_mode_2        str          78482            27.80   

                          high missing  
fatality_mode_1                  False  
fatality_mode_2                  False  
seriously_injured_mode_1         False  
seriously_injured_mode_2         False  


In [15]:
df.drop(columns = ["fatality_mode_1", "fatality_mode_2",
                   "seriously_injured_mode_1", "seriously_injured_mode_2"], inplace = True)

In [16]:
df.head(20)

,top_traffic_accident_offense,first_occurrence_date,incident_address,geo_lon,geo_lat,tu1_driver_action,tu2_driver_action,tu1_vehicle_type,tu2_vehicle_type,tu1_driver_humancontribfactor,...,seriously_injured,fatalities,tu2_pedestrian_action,road_description,road_condition,light_condition,datetime,date,time,injured
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,no contributing action,no contributing action,pickup truck/utility van,transit bus,no apparent contributing factor,...,0.0,0.0,NaN,non-intersection,dry,daylight,2022-12-09 02:05:59.999991956,2022-12-09,02:05:59,0
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,failed to yield row,no contributing action,passenger car/passenger van,suv,no apparent contributing factor,...,0.0,0.0,NaN,at intersection,dry,daylight,2022-12-11 23:39:59.999991056,2022-12-11,23:39:59,0
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,followed too closely,no contributing action,passenger car/passenger van,passenger car/passenger van,no apparent contributing factor,...,0.0,0.0,NaN,non-intersection,dry,daylight,2022-12-12 00:10:00.000004471,2022-12-12,00:10:00,0
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,followed too closely,no contributing action,pickup truck/utility van,passenger car/passenger van,aggressive driving,...,0.0,0.0,NaN,non-intersection,dry,dark-lighted,2022-12-09 06:26:59.999983903,2022-12-09,06:26:59,0
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,<NA>,<NA>,unknown (hit and run only),passenger car/passenger van,<NA>,...,0.0,0.0,NaN,non-intersection,dry,dark-lighted,2022-12-09 07:30:00.000000000,2022-12-09,07:30:00,0
5,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,S OSCEOLA ST / W BAYAUD AVE,-105.038499,39.714959,careless driving,no contributing action,passenger car/passenger van,passenger car/passenger van,not observed,...,0.0,0.0,NaN,non-intersection,dry,dark-lighted,2022-12-02 14:00:00.000013415,2022-12-02,14:00:00,0
6,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,300 BLOCK E 17TH AVE,-104.983103,39.743008,<NA>,no contributing action,<NA>,passenger car/passenger van,<NA>,...,0.0,0.0,NaN,non-intersection,dry,daylight,2022-12-03 00:20:00.000008943,2022-12-03,00:20:00,0
7,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,N OGDEN ST / E 11TH AVE,-104.975278,39.733689,disregarded stop sign,no contributing action,pickup truck/utility van,passenger car/passenger van,no apparent contributing factor,...,0.0,0.0,NaN,at intersection,dry,dark-lighted,2022-12-03 03:59:59.999986584,2022-12-03,03:59:59,0
8,TRAF - ACCIDENT,2.459916e+06,N CENTRAL PARK BLVD / N VERBENA ST,-104.889926,39.751852,careless driving,lane violation,passenger car/passenger van,passenger car/passenger van,aggressive driving,...,0.0,0.0,NaN,intersection related,dry,daylight,2022-12-03 03:45:00.000000000,2022-12-03,03:45:00,0
9,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,I70 HWYWB / N NORTHFIELD QUEBEC ST,-104.903436,39.778421,lane violation,no contributing action,passenger car/passenger van,passenger car/passenger van,not observed,...,0.0,0.0,NaN,intersection related,dry,dark-lighted,2022-12-03 07:29:00.000003572,2022-12-03,07:29:00,0


## Recoding

The following cells recode the messy text columns to understand the columns and combine classes. For example, distracted coveres "distracted-other", "distracted/other interior", "distracted/other exterior". 

This is completed for the following sets of columns 

| Information | Column names | Descrip. |  Example|
| -------- | -------- | -------- | -------- |
| vehicle_type | `tu1_vehicle_type`, `tu2_vehicle_type` | Vehicle Type |  "passenger", "vehicle over 10000 lbs" |
| driver actions | `tu1_driver_action`, `tu2_driver_action` | Driver actions, specifically in vehicle operation | "failure to yeild" |
| human factor | `tu1_driver_humancontribfactor`, `tu2_driver_humancontribfactor` | Driver actions, not directly in vehicle movement | "distracted", "medical" |

For each pair of columns, I 
1. count how often each entry appears per column (`create_types_table`)
2. combine related entries (`bin_`) 
3. try to understand how many of each accident type, based on binned columns, were injurious. 

### Vehicle Types 

In [17]:
vehicle_types = create_types_tables(
    df,
    "tu1_vehicle_type",
    "tu2_vehicle_type",
    first_label="first_vehicle_type",
    second_label="second_vehicle_type"
)

vehicle_types

,first_vehicle_type,second_vehicle_type,total
passenger car/van,96818,97346,194164
suv,59399,64388,123787
passenger car/passenger van,34815,37401,72216
pickup truck/utility van,31487,28162,59649
other,8104,16253,24357
<NA>,6849,10037,16886
hit and run unknown,14702,483,15185
,2130,8946,11076
vehicle over 10000 lbs,6324,4630,10954
pickup truck/utility van with trailier,4680,3866,8546


In [18]:
df["tu1_vehicle_type_binned"] = df["tu1_vehicle_type"].apply(bin_vehicle_type)
df["tu2_vehicle_type_binned"] = df["tu2_vehicle_type"].apply(bin_vehicle_type)

In [19]:
accident_summary_wrt(df, col= "tu1_vehicle_type_binned")

,count,mean
tu1_vehicle_type_binned,,
motorcycle_or_autocycle,1988,0.348592
bicycle_or_motorized_bicycle,1221,0.156429
other,8460,0.058156
equipment_or_special_vehicle,255,0.054902
passenger_car_or_van,131994,0.021933
suv,59577,0.019722
pickup_or_utility_van,36167,0.018221
heavy_truck,8316,0.012145
bus,2120,0.010849


### Driver actions

In [20]:
driver_action = create_types_tables(
    df,
    "tu1_driver_action",
    "tu2_driver_action",
    first_label="first_driver_action",
    second_label="first_driver_action"
)

driver_action

,first_driver_action,total
tu2_driver_action,,
other,91523,183046
no action,63197,126394
no contributing action,61369,122738
00,29025,58050
<NA>,15145,30290
,14136,28272
under investigation,1617,3234
lane violation,1106,2212
followed too closely,969,1938


In [21]:
df["tu1_driver_action_binned"] = (df["tu1_driver_action"].apply(bin_driver_action))
df["tu2_driver_action_binned"] = (df["tu2_driver_action"].apply(bin_driver_action))

In [22]:
accident_summary_wrt(df, col= "tu1_driver_action_binned")

,count,mean
tu1_driver_action_binned,,
traffic_flow,323,0.043344
failure_to_obey,15110,0.036400
speed_related,5023,0.030460
failure_to_yield,27769,0.029097
aggressive_or_careless,104774,0.028986
other,53777,0.022314
no_action,14694,0.015448
unknown,20828,0.012099
lane_or_position,26150,0.008298


### Human contributing factor

In [23]:
human_factor = create_types_tables(
    df,
    "tu1_driver_humancontribfactor",
    "tu2_driver_humancontribfactor",
    first_label="first_driver",
    second_label="second_driver"
)

human_factor

,first_driver,second_driver,total
no apparent,78474,158518,236992
no apparent contributing factor,16372,60315,76687
other,37660,26067,63727
<NA>,15813,14181,29994
aggressive driving,23841,912,24753
,8808,15310,24118
distracted-other,16721,324,17045
driver inexperience,15977,525,16502
not observed,12798,2813,15611
dui/dwai/duid,10710,186,10896


In [24]:
df['tu1_human_factor_binned'] = (df["tu1_driver_humancontribfactor"]
                        .apply(bin_human_contrib_factor))

df['tu2_human_factor_binned'] = (df["tu2_driver_humancontribfactor"]
                        .apply(bin_human_contrib_factor))

In [25]:
accident_summary_wrt(df, col= "tu1_human_factor_binned")

,count,mean
tu1_human_factor_binned,,
medical_or_ability,3786,0.061014
impairment,10710,0.056489
aggressive_or_evading,24639,0.043184
environmental_visibility,354,0.031073
fatigue_or_sleep,3225,0.029457
other,41830,0.026584
emotional,759,0.023715
inexperience_or_unfamiliar,25047,0.021280
distracted,34932,0.019953


Among the 3,787 crashes where driver 1 had a medical/ability-related contributing factor, about 6.1% resulted in injury/fatality.

However, this does not mean the medical factors caused the accident or caused the accident top be injurious. Of the crashed that could be categorized, 6% involved injury. 

In [26]:
df.drop(columns = ["tu1_driver_action", "tu2_driver_action",
                   "tu1_vehicle_type", "tu2_vehicle_type",
                   'tu1_driver_humancontribfactor',
                   'tu2_driver_humancontribfactor',
                   'tu1_pedestrian_action',
                   'tu2_pedestrian_action',
                   "seriously_injured",
                   "fatalities"], inplace=True)

df = df.rename(columns= {
    "geo_lon" : "lon",
    "geo_lat" : 'lat'
})

In [27]:
df.head(25)

,top_traffic_accident_offense,first_occurrence_date,incident_address,lon,lat,road_description,road_condition,light_condition,datetime,date,time,injured,tu1_vehicle_type_binned,tu2_vehicle_type_binned,tu1_driver_action_binned,tu2_driver_action_binned,tu1_human_factor_binned,tu2_human_factor_binned
0,TRAF - ACCIDENT,2.459922e+06,I25 HWYNB / W 6TH AVE,-105.013229,39.725677,non-intersection,dry,daylight,2022-12-09 02:05:59.999991956,2022-12-09,02:05:59,0,pickup_or_utility_van,bus,no_action,no_action,no_apparent,no_apparent
1,TRAF - ACCIDENT,2.459925e+06,E 17TH AVE / N PENNSYLVANIA ST,-104.981084,39.743271,at intersection,dry,daylight,2022-12-11 23:39:59.999991056,2022-12-11,23:39:59,0,passenger_car_or_van,suv,failure_to_yield,no_action,no_apparent,no_apparent
2,TRAF - ACCIDENT,2.459925e+06,I25 HWYSB / PARK AVEW,-104.994972,39.768330,non-intersection,dry,daylight,2022-12-12 00:10:00.000004471,2022-12-12,00:10:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,no_apparent,no_apparent
3,TRAF - ACCIDENT - POLICE,2.459922e+06,I25 HWYNB / 20TH ST,-105.004334,39.760949,non-intersection,dry,dark-lighted,2022-12-09 06:26:59.999983903,2022-12-09,06:26:59,0,pickup_or_utility_van,passenger_car_or_van,aggressive_or_careless,no_action,aggressive_or_evading,no_apparent
4,TRAF - ACCIDENT - HIT & RUN,2.459922e+06,W 37TH AVE / N PECOS ST,-105.006450,39.768054,non-intersection,dry,dark-lighted,2022-12-09 07:30:00.000000000,2022-12-09,07:30:00,0,unknown,passenger_car_or_van,unknown,unknown,unknown,unknown
5,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,S OSCEOLA ST / W BAYAUD AVE,-105.038499,39.714959,non-intersection,dry,dark-lighted,2022-12-02 14:00:00.000013415,2022-12-02,14:00:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,no_action,unknown,no_apparent
6,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,300 BLOCK E 17TH AVE,-104.983103,39.743008,non-intersection,dry,daylight,2022-12-03 00:20:00.000008943,2022-12-03,00:20:00,0,unknown,passenger_car_or_van,unknown,no_action,unknown,no_apparent
7,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,N OGDEN ST / E 11TH AVE,-104.975278,39.733689,at intersection,dry,dark-lighted,2022-12-03 03:59:59.999986584,2022-12-03,03:59:59,0,pickup_or_utility_van,passenger_car_or_van,failure_to_obey,no_action,no_apparent,no_apparent
8,TRAF - ACCIDENT,2.459916e+06,N CENTRAL PARK BLVD / N VERBENA ST,-104.889926,39.751852,intersection related,dry,daylight,2022-12-03 03:45:00.000000000,2022-12-03,03:45:00,0,passenger_car_or_van,passenger_car_or_van,aggressive_or_careless,lane_or_position,aggressive_or_evading,aggressive_or_evading
9,TRAF - ACCIDENT - HIT & RUN,2.459916e+06,I70 HWYWB / N NORTHFIELD QUEBEC ST,-104.903436,39.778421,intersection related,dry,dark-lighted,2022-12-03 07:29:00.000003572,2022-12-03,07:29:00,0,passenger_car_or_van,passenger_car_or_van,lane_or_position,no_action,unknown,no_apparent


In [28]:
df.to_csv("data/interim/traffic_clean.csv", index = False)